# CLIO: Smartphone product classification and review count analysis

This notebook screens Amazon Reviews 2023 metadata for phone-related products, classifies the candidates, evaluates predictions against existing human labels, prepares a stratified review sample, and streams the review file once to count rows for selected smartphones. The `rating_number >= 100` rule controls the analysis population; it is not itself evidence that a product is a smartphone.

## A — Setup and configuration

Resolve paths from the repository rather than a user-specific absolute path. Existing manually labelled CSVs are loaded before metadata processing and are only updated after predictions are ready.

In [108]:
import gzip
import hashlib
import json
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (expected data/raw/ and notebooks/)")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REVIEW_PATH = RAW_DIR / "Cell_Phones_and_Accessories.jsonl.gz"
METADATA_PATH = RAW_DIR / "meta_Cell_Phones_and_Accessories.jsonl.gz"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INDEPENDENT_PATH = PROCESSED_DIR / "smartphone_filter_independent_validation.csv"
STRATIFIED_PATH = PROCESSED_DIR / "smartphone_filter_stratified_validation.csv"
CLASSIFICATION_PATH = PROCESSED_DIR / "smartphone_product_classification.csv"
COUNTS_PATH = PROCESSED_DIR / "smartphone_review_counts.csv"
DIAGNOSTIC_PATH = PROCESSED_DIR / "smartphone_review_count_diagnostic.csv"

for path in (REVIEW_PATH, METADATA_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required input file is missing: {path}")

print("Project root:", PROJECT_ROOT)
print("Metadata:", METADATA_PATH, f"({METADATA_PATH.stat().st_size:,} bytes)")
print("Reviews:", REVIEW_PATH, f"({REVIEW_PATH.stat().st_size:,} bytes)")
print("Processed output directory:", PROCESSED_DIR)
print("Existing independent validation:", INDEPENDENT_PATH.is_file())
print("Existing stratified review file:", STRATIFIED_PATH.is_file())

# Load existing label files before classification. Never replace the human columns.
independent_sample = pd.read_csv(INDEPENDENT_PATH, dtype={"parent_asin":"string", "asin":"string"}) if INDEPENDENT_PATH.is_file() else None
stratified_sample = pd.read_csv(STRATIFIED_PATH, dtype={"parent_asin":"string", "asin":"string"}) if STRATIFIED_PATH.is_file() else None
if independent_sample is not None:
    print("Independent sample rows:", len(independent_sample), "columns:", independent_sample.columns.tolist())
if stratified_sample is not None:
    print("Stratified sample rows:", len(stratified_sample), "columns:", stratified_sample.columns.tolist())

SAMPLE_IDS = set()
for sample in (independent_sample, stratified_sample):
    if sample is not None and "parent_asin" in sample:
        SAMPLE_IDS.update(sample["parent_asin"].dropna().astype(str).str.strip())
SAMPLE_IDS.discard("")

CLASS_ORDER = ["smartphone", "feature_phone", "accessory", "needs_review"]
CLASSIFIER_VERSION = "notebook03-rule-v1.2"

Project root: /Users/vaishnavipoojary/CliO-customer-feedback-analytics
Metadata: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/raw/meta_Cell_Phones_and_Accessories.jsonl.gz (857,990,660 bytes)
Reviews: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/raw/Cell_Phones_and_Accessories.jsonl.gz (2,532,539,010 bytes)
Processed output directory: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed
Existing independent validation: True
Existing stratified review file: True
Independent sample rows: 100 columns: ['parent_asin', 'title', 'average_rating', 'rating_number', 'manual_label', 'v2_prediction', 'reviewer_notes', 'predicted_class', 'classifier_version']
Stratified sample rows: 153 columns: ['parent_asin', 'title', 'rating_number', 'classifier_prediction', 'manual_label', 'reviewer_notes', 'predicted_class', 'classifier_version']


## B — Understand metadata and C — Select candidates

The metadata archive is streamed once. The scan counts valid/malformed lines, unique product identifiers (parent ASIN with ASIN fallback), repeated identifiers, missing IDs and titles, observed fields, and representative titles. The deterministic candidate pool requires `rating_number >= 100` plus phone-related title evidence. Product classification below separates likely phones from accessories; the threshold does not label a product.

In [109]:
PHONE_TITLE = re.compile(
    r"\b(?:smart\s?phone|cell\s?phone|mobile\s?phone|iphone|google pixel|pixel\s+\d|"
    r"samsung galaxy|galaxy\s+[sazm]\s?\d|galaxy note|galaxy fold|galaxy flip|"
    r"oneplus|xiaomi|redmi|poco|motorola|moto\s+[a-z0-9]+|nokia|oppo|vivo|realme|"
    r"huawei|honor|alcatel|sony xperia|tecno|infinix|blu vivo|blu studio|"
    r"unlocked phone|prepaid phone|feature phone|senior phone)\b", re.I
)


def product_identifier(product):
    """Return parent ASIN, falling back to ASIN for product-level metadata inventory."""
    return str(product.get("parent_asin") or product.get("asin") or "").strip()


def numeric_rating_count(value):
    try:
        number = float(value)
        return number if pd.notna(number) else None
    except (TypeError, ValueError):
        return None


metadata_lines = 0
metadata_valid = 0
metadata_malformed = 0
missing_identifier_count = 0
missing_title_count = 0
unique_metadata_ids = set()
metadata_duplicate_ids = set()
metadata_fields = set()
metadata_title_examples = []
candidate_records = []
candidate_parent_ids = set()
candidate_duplicate_rows = 0
candidate_missing_parent_ids = 0
sample_metadata = {}

with gzip.open(METADATA_PATH, "rt", encoding="utf-8") as stream:
    for line in stream:
        metadata_lines += 1
        try:
            product = json.loads(line)
        except json.JSONDecodeError:
            metadata_malformed += 1
            continue
        if not isinstance(product, dict):
            metadata_malformed += 1
            continue
        metadata_valid += 1
        metadata_fields.update(product.keys())
        identifier = product_identifier(product)
        if not identifier:
            missing_identifier_count += 1
        elif identifier in unique_metadata_ids:
            metadata_duplicate_ids.add(identifier)
        else:
            unique_metadata_ids.add(identifier)
        title = product.get("title")
        if not isinstance(title, str) or not title.strip():
            missing_title_count += 1
        elif len(metadata_title_examples) < 12:
            metadata_title_examples.append(title.strip())

        parent = str(product.get("parent_asin") or "").strip()
        if parent and parent in SAMPLE_IDS and parent not in sample_metadata:
            sample_metadata[parent] = product

        rating_count = numeric_rating_count(product.get("rating_number"))
        if rating_count is None or rating_count < 100 or not isinstance(title, str) or not PHONE_TITLE.search(title):
            continue
        if not parent:
            candidate_missing_parent_ids += 1
            # Keep a record with whatever product identifier exists; it cannot join by parent later.
            candidate_key = identifier or f"missing-parent-row-{metadata_lines}"
        else:
            candidate_key = parent
        if candidate_key in candidate_parent_ids:
            candidate_duplicate_rows += 1
            continue
        candidate_parent_ids.add(candidate_key)
        candidate_records.append({
            "parent_asin": parent,
            "asin": str(product.get("asin") or "").strip(),
            "title": title.strip(),
            "average_rating": product.get("average_rating"),
            "rating_number": product.get("rating_number"),
            "main_category": product.get("main_category"),
            "categories": product.get("categories"),
            "features": product.get("features"),
            "description": product.get("description"),
            "details": product.get("details"),
        })
        if metadata_lines % 1_000_000 == 0:
            print(f"Metadata lines read: {metadata_lines:,}")

print("Metadata lines:", f"{metadata_lines:,}")
print("Valid metadata records:", f"{metadata_valid:,}")
print("Malformed metadata records:", f"{metadata_malformed:,}")
print("Unique product identifiers:", f"{len(unique_metadata_ids):,}")
print("Identifiers repeated at least once:", f"{len(metadata_duplicate_ids):,}")
print("Records missing product identifier:", f"{missing_identifier_count:,}")
print("Records missing title:", f"{missing_title_count:,}")
print("Available metadata fields:", sorted(metadata_fields))
print("Representative titles:")
for title in metadata_title_examples:
    print(" •", title)
print("\nCandidates (rating_number >= 100 and phone-related title):", f"{len(candidate_records):,}")
print("Duplicate candidate rows skipped:", f"{candidate_duplicate_rows:,}")
print("Candidate rows missing parent_asin:", f"{candidate_missing_parent_ids:,}")
candidate_metadata_lookup = {str(record["parent_asin"]): record for record in candidate_records if record["parent_asin"]}
print("Existing validation sample metadata matched:", f"{len(sample_metadata):,} / {len(SAMPLE_IDS):,}")

Metadata lines: 1,288,490
Valid metadata records: 1,288,490
Malformed metadata records: 0
Unique product identifiers: 1,288,490
Identifiers repeated at least once: 0
Records missing product identifier: 0
Records missing title: 71
Available metadata fields: ['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
Representative titles:
 • ARAREE Slim Diary Cell Phone Case for Samsung Galaxy Note 5 - Retail Packaging - Coral Pink
 • Bastmei for OnePlus 7T Case Extremely Light Ultra Thin, Super Slim Hard PC Cover Case for OnePlus 7T (Gravel Green)
 • Wireless Fones Branded New Iphone 5C/LITE Hot Pink Lace detail Leather Wallet Folio Case
 • iPhone 6 Plus + Case, DandyCase Perfect PATTERNNo Chip/No Peel Flexible Slim TPU Case Cover for Apple iPhone 6 Plus (5.5" Screen) [Vintage Pink Rose Floral]
 • Case for Galaxy S6/S6 Edge, Thin Translucen

## D — Classify products

`classify_product_v2(product)` returns one of the four documented classes. Explicit accessory evidence is checked before phone model names so compatibility references do not turn accessories into phones. Missing or conflicting evidence defaults to `needs_review`. The small synthetic checks below verify code behavior only; they are not dataset validation.

In [110]:
ACCESSORY_TITLE = re.compile(
    r"\b(?:smartphone mount|smartphone holder|mount holder|watch bands?|wristbands?|screen\s+(?:[a-z0-9]+\s+){0,2}glass.*\breplacement\b|screen film|case|cover|screen protector|screen guard|tempered glass|charger|charging cable|"
    r"usb cable|adapter|power bank|phone mount|phone holder|phone stand|car mount|"
    r"replacement screen|replacement display|digitizer|lcd assembly|battery replacement|"
    r"camera lens|lens protector|phone skin|phone grip|pop\s?socket|ring holder|"
    r"gimbal|stabilizer|selfie stick|tripod|phone cage|phone fan|headset|earbuds|"
    r"headphones|stylus|holster|belt clip|lanyard|screen amplifier|phone charm)\b", re.I
)
ACCESSORY_CATEGORIES = re.compile(
    r"\b(?:cases|screen protectors|chargers|cables|adapters|mounts|holders|stands|"
    r"replacement parts|phone accessories|cell phone accessories|camera lenses)\b", re.I
)
FEATURE_PHONE = re.compile(r"\b(?:feature phone|senior phone|series\s?40|nokia\s?(?:3310|6300|105|110|225|2720)|motorola\s?(?:i580|razr v3))\b", re.I)
SMARTPHONE_MODEL = re.compile(
    r"\b(?:iphone(?:\s?\d|\s?se|\s?x)|google pixel\s?[0-9a-z]|pixel\s?[0-9][a-z]?|"
    r"samsung galaxy\s?(?:[sazm]\s?\d|note\s?\d|fold\s?\d|flip\s?\d)|"
    r"galaxy\s?(?:[sazm]\s?\d|note\s?\d)|oneplus\s?(?:\d|one)|redmi\s?(?:note\s?)?\d|"
    r"poco\s?[a-z0-9]+|xiaomi\s?(?:mi|redmi|poco)|motorola\s?(?:edge|razr)|moto\s?g\s?\d|"
    r"sony xperia\s?[a-z0-9]+|nokia\s?(?:[gx]\s?\d|lumia)|oppo\s?(?:find|reno)|"
    r"vivo\s?[xv]\s?\d|realme\s?\d|huawei\s?(?:p|mate|nova|y)\s?\d|honor\s?(?:magic|\d)|"
    r"alcatel\s?(?:one touch|pixi)|blu\s?(?:vivo|studio|g\s?\d)|htc\s?(?:one|u\s?\d))\b", re.I
)


def flatten_text(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        return " ".join(f"{k} {flatten_text(v)}" for k, v in value.items())
    if isinstance(value, (list, tuple, set)):
        return " ".join(flatten_text(v) for v in value)
    return str(value)


def classify_product_v2(product):
    """Rule-based metadata classifier; ambiguous products deliberately remain needs_review."""
    if not isinstance(product, dict):
        return "needs_review"
    title = str(product.get("title") or "").strip()
    if not title:
        return "needs_review"
    categories = flatten_text(product.get("categories"))
    details = product.get("details") if isinstance(product.get("details"), dict) else {}
    form_factor = " ".join(str(v) for k, v in details.items() if "form factor" in str(k).lower()).lower()
    title_and_meta = title + " " + flatten_text(product.get("features"))

    # Compatibility mentions are not a device signal. Concrete accessory type wins.
    if ACCESSORY_TITLE.search(title) or ACCESSORY_CATEGORIES.search(categories) or form_factor in {"case", "bumper"}:
        return "accessory"
    if FEATURE_PHONE.search(title) or "series 40" in flatten_text(details).lower():
        return "feature_phone"

    # Use dedicated OS/form-factor fields only; general marketing copy can mention Android.
    device_evidence = []
    for key, value in product.items():
        if str(key).lower() in {"operating_system", "operating system", "os", "phone_type", "device_type"}:
            device_evidence.append(flatten_text(value).lower())
    if any(re.search(r"\b(android|ios|smartphone|smart phone)\b", value) for value in device_evidence):
        return "smartphone"
    if SMARTPHONE_MODEL.search(title):
        return "smartphone"
    # A plain smartphone term is sufficient only when the listing is not a compatibility item.
    if re.search(r"\b(?:unlocked )?smart ?phone\b", title, re.I) and not re.search(r"\b(?:for|fits|compatible with)\b", title, re.I):
        return "smartphone"
    return "needs_review"


synthetic_cases = [
    ({"title":"Samsung Galaxy S21 5G Smartphone"}, "smartphone"),
    ({"title":"Nokia 3310 Feature Phone"}, "feature_phone"),
    ({"title":"Protective phone case for iPhone 14"}, "accessory"),
    ({"title":"Tempered glass screen protector for Samsung Galaxy S22"}, "accessory"),
    ({"title":"Replacement watch band for smartwatch"}, "accessory"),
    ({"title":"Screen S7 Glass replacement"}, "accessory"),
    ({"title":"Universal phone holder compatible with iPhone"}, "accessory"),
    ({"title":"Universal Smartphone Mount Holder with Clamp"}, "accessory"),
    ({"title":"Smartphone gimbal stabilizer"}, "accessory"),
    ({"title":"Unknown item"}, "needs_review"),
    ({"title":"", "details":None}, "needs_review"),
    ({"title":None}, "needs_review"),
]
for item, expected in synthetic_cases:
    actual = classify_product_v2(item)
    assert actual == expected, (item, expected, actual)
assert all(classify_product_v2(item) in CLASS_ORDER for item, _ in synthetic_cases)
print(f"Synthetic classifier behavior checks passed: {len(synthetic_cases)} examples")

Synthetic classifier behavior checks passed: 12 examples


## E — Classify the selected candidate pool

The classification CSV contains unique metadata candidates meeting both `rating_number >= 100` and the broad phone-title retrieval rule. It is not the complete category catalog. Optional metadata fields may be blank; rows are retained. Product IDs are written as strings.

In [111]:
classification_rows = []
for product in candidate_records:
    classification_rows.append({
        "parent_asin": product["parent_asin"],
        "asin": product["asin"],
        "title": product["title"],
        "average_rating": product["average_rating"],
        "rating_number": product["rating_number"],
        "predicted_class": classify_product_v2(product),
        "classifier_version": CLASSIFIER_VERSION,
    })
classification = pd.DataFrame(classification_rows)
if classification.empty:
    raise ValueError("Candidate set is empty; check the metadata input and selection rules")
for col in ("parent_asin", "asin"):
    classification[col] = classification[col].fillna("").astype(str)

class_counts = classification["predicted_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
class_summary = pd.DataFrame({"predicted_class": CLASS_ORDER, "count": class_counts.values})
class_summary["percent"] = class_summary["count"] / len(classification) * 100
print("Predicted class counts and percentages:")
display(class_summary)
print("Products requiring manual review:", int(class_counts["needs_review"]))
for label in CLASS_ORDER:
    print(f"\nRepresentative {label} titles:")
    display(classification.loc[classification.predicted_class.eq(label), ["parent_asin", "title"]].head(5))
classification.to_csv(CLASSIFICATION_PATH, index=False)
print("Saved:", CLASSIFICATION_PATH, "rows:", len(classification))

Predicted class counts and percentages:


,predicted_class,count,percent
0,smartphone,2768,1.924307
1,feature_phone,46,0.031979
2,accessory,137354,95.488168
3,needs_review,3676,2.555546


Products requiring manual review: 3676

Representative smartphone titles:


,parent_asin,title
12,B00CP2YPYI,Puregear 60184PG Dualtek for Galaxy S4 - 1 Pac...
51,B07SDQ483V,"BLU Vivo XL4 6.2"" HD Display Smartphone 32Gb+3..."
150,B076CCY2D3,Huawei Mate 10 Pro (Single-SIM) 128GB BLA-L09 ...
339,B07YHMHWQD,"UMIDIGI A11S Unlocked Cell Phone, 6.53"" FHD Fu..."
535,B00D1A37OM,"Nokia Lumia 810 T-Mobile Phone, 4.3-Inch. 8GB ..."



Representative feature_phone titles:


...


Representative accessory titles:


,parent_asin,title
0,B07ZPSG8P5,Bastmei for OnePlus 7T Case Extremely Light Ul...
1,B0BHY2CR5B,"iMangoo Galaxy S22 Case, for Samsung Galaxy S2..."
2,B08GK7J9XN,ZHK Compatible with iPhone 12 Pro Max Case for...
3,B0B8ZBCCY6,WSKEN for Samsung Galaxy Z Fold 4 Camera Lens ...
4,B07NWTB6P1,MMOBIEL Earspeaker Light Sensor Flex Cable Inc...



Representative needs_review titles:


,parent_asin,title
8,B09QC96CDF,Areziir Gapless Elegant Bands Compatible with ...
78,B098BK2SJ5,SAMSUNG Galaxy A52 | A525F | 128GB 6GB RAM | F...
85,B01GINVN1W,Moto G (4th Generation) - Black - 32 GB - Unlo...
95,B07HHG6CXS,OnePlus 6T A6013 Dual Sim 128GB/8GB (Mirror Bl...
117,B0BP75J2BH,Eweison Compatible with Samsung Galaxy Watch 4...


Saved: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed/smartphone_product_classification.csv rows: 143844


## F — Evaluate the existing independent sample

Existing labels and notes remain untouched. Predictions are refreshed from the current classifier using metadata captured in the initial stream. Blank labels and manual `needs_review` are excluded from definitive metrics. A classifier `needs_review` prediction remains a valid prediction and is a mismatch for a definitive human label.

In [112]:
def evaluate_manual_labels(frame, manual_col="manual_label", prediction_col="predicted_class", min_support=10):
    allowed = set(CLASS_ORDER)
    manual = frame[manual_col].fillna("").astype(str).str.strip()
    predicted = frame[prediction_col].fillna("").astype(str).str.strip()
    invalid_manual = sorted(set(manual) - allowed - {""})
    invalid_pred = sorted(set(predicted) - allowed - {""})
    if invalid_manual:
        raise ValueError(f"Invalid manual labels in {manual_col}: {invalid_manual}")
    if invalid_pred:
        raise ValueError(f"Invalid predictions in {prediction_col}: {invalid_pred}")
    eligible = manual.ne("") & manual.ne("needs_review")
    truth = manual[eligible]
    pred = predicted[eligible].replace("", "needs_review")
    matrix = pd.DataFrame(0, index=CLASS_ORDER, columns=CLASS_ORDER, dtype=int)
    for actual, guess in zip(truth, pred):
        matrix.loc[actual, guess] += 1
    report_rows = []
    for label in CLASS_ORDER:
        tp = int(matrix.loc[label, label])
        support = int(matrix.loc[label].sum())
        predicted_n = int(matrix[label].sum())
        precision = tp / predicted_n if predicted_n else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        report_rows.append({"class":label,"precision":precision,"recall":recall,"f1_score":f1,"support":support,"insufficient_support":support < min_support})
    eligible_n = int(eligible.sum())
    matches = int((truth.to_numpy() == pred.to_numpy()).sum())
    mismatches = frame.loc[eligible & manual.ne(predicted)].copy()
    return {
        "total_rows":len(frame), "eligible_rows":eligible_n,
        "blank_manual_labels":int(manual.eq("").sum()),
        "uncertain_manual_labels":int(manual.eq("needs_review").sum()),
        "manual_distribution":manual.value_counts(dropna=False),
        "prediction_distribution":predicted.value_counts(dropna=False),
        "accuracy":matches / eligible_n if eligible_n else None,
        "report":pd.DataFrame(report_rows).set_index("class"),
        "confusion_matrix":matrix,
        "mismatches":mismatches,
    }


def add_sample_predictions(sample, name):
    sample = sample.copy()
    if "parent_asin" not in sample:
        raise ValueError(f"{name} sample is missing parent_asin")
    if "manual_label" not in sample:
        sample["manual_label"] = ""
    if "reviewer_notes" not in sample:
        sample["reviewer_notes"] = ""
    predictions = {}
    for parent in sample["parent_asin"].fillna("").astype(str).str.strip().unique():
        product = sample_metadata.get(parent) or candidate_metadata_lookup.get(parent)
        predictions[parent] = classify_product_v2(product) if product else "needs_review"
    sample["predicted_class"] = sample["parent_asin"].fillna("").astype(str).str.strip().map(predictions)
    if "v2_prediction" in sample.columns:
        sample["v2_prediction"] = sample["predicted_class"]
    sample["classifier_version"] = CLASSIFIER_VERSION
    return sample

if independent_sample is not None:
    independent_sample = add_sample_predictions(independent_sample, "independent")
    result = evaluate_manual_labels(independent_sample)
    print("Independent sample rows:", result["total_rows"])
    print("Definitively labelled:", result["eligible_rows"], "blank:", result["blank_manual_labels"], "manual needs_review:", result["uncertain_manual_labels"])
    print("Manual class distribution:"); display(result["manual_distribution"])
    print("Predicted class distribution:"); display(result["prediction_distribution"])
    print("Accuracy:", result["accuracy"])
    display(result["report"]); display(result["confusion_matrix"])
    print("Insufficient support classes:", result["report"].index[result["report"]["insufficient_support"]].tolist())
    print("Mismatches:"); display(result["mismatches"][[c for c in ["parent_asin","title","manual_label","predicted_class"] if c in result["mismatches"]]])
    independent_sample.to_csv(INDEPENDENT_PATH, index=False)
    print("Updated predictions while preserving human columns:", INDEPENDENT_PATH)
else:
    print("Independent validation file not found; no human-labeled independent sample is available. No labels were fabricated.")

Independent sample rows: 100
Definitively labelled: 92 blank: 0 manual needs_review: 8
Manual class distribution:


manual_label
accessory       91
needs_review     8
smartphone       1
Name: count, dtype: int64

Predicted class distribution:


predicted_class
accessory       96
needs_review     3
smartphone       1
Name: count, dtype: int64

Accuracy: 0.9891304347826086


,precision,recall,f1_score,support,insufficient_support
class,,,,,
smartphone,1.0,1.000000,1.000000,1,True
feature_phone,0.0,0.000000,0.000000,0,True
accessory,1.0,0.989011,0.994475,91,False
needs_review,0.0,0.000000,0.000000,0,True


,smartphone,feature_phone,accessory,needs_review
smartphone,1,0,0,0
feature_phone,0,0,0,0
accessory,0,0,90,1
needs_review,0,0,0,0


Insufficient support classes: ['smartphone', 'feature_phone', 'needs_review']
Mismatches:


,parent_asin,title,manual_label,predicted_class
0,B01D8HMJFA,MaxDemo M46 Screen S7 Glass 112390,accessory,needs_review


Updated predictions while preserving human columns: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed/smartphone_filter_independent_validation.csv


## G — Prepare stratified manual review and H — Score it

If the existing stratified CSV is available, preserve its rows, manual labels, and reviewer notes while refreshing prediction columns. If it is missing, sample up to 50 products per predicted class with a fixed seed. Newly sampled labels and notes remain blank. Rerun the scoring cell after manual labeling.

In [113]:
if stratified_sample is None:
    random.seed(2026)
    stratified_parts = []
    for label in CLASS_ORDER:
        group = classification.loc[classification.predicted_class.eq(label)]
        if len(group):
            n = min(50, len(group))
            stratified_parts.append(group.sample(n=n, random_state=2026))
    if not stratified_parts:
        raise ValueError("No candidate products available for stratified manual review")
    stratified_sample = pd.concat(stratified_parts, ignore_index=True)
    stratified_sample["manual_label"] = ""
    stratified_sample["reviewer_notes"] = ""

stratified_sample = add_sample_predictions(stratified_sample, "stratified")
# Retain the existing legacy prediction column too, if it was part of the file schema.
stratified_sample["classifier_prediction"] = stratified_sample["predicted_class"]
stratified_sample.to_csv(STRATIFIED_PATH, index=False)
print("Stratified review sample rows:", len(stratified_sample))
print("Predicted strata:"); display(stratified_sample["predicted_class"].value_counts().reindex(CLASS_ORDER, fill_value=0))
print("Manual labels filled:", int(stratified_sample["manual_label"].fillna("").astype(str).str.strip().ne("").sum()))
print("Saved with human labels/notes preserved:", STRATIFIED_PATH)

stratified_for_scoring = pd.read_csv(STRATIFIED_PATH, dtype={"parent_asin":"string"})
stratified_for_scoring["predicted_class"] = stratified_for_scoring.get("predicted_class", stratified_for_scoring.get("classifier_prediction", ""))
stratified_result = evaluate_manual_labels(stratified_for_scoring)
print("Stratified sample rows:", stratified_result["total_rows"])
print("Definitive labels eligible:", stratified_result["eligible_rows"])
print("Blank labels excluded:", stratified_result["blank_manual_labels"])
print("Manual needs_review labels excluded:", stratified_result["uncertain_manual_labels"])
if stratified_result["eligible_rows"] == 0:
    print("No completed manual labels; evaluation cannot yet be performed.")
else:
    print("Accuracy:", stratified_result["accuracy"])
    display(stratified_result["report"]); display(stratified_result["confusion_matrix"])
    print("Mismatches:"); display(stratified_result["mismatches"])
print("Insufficient support classes:", stratified_result["report"].index[stratified_result["report"]["insufficient_support"]].tolist())

# Small synthetic checks for scorer semantics and fixed confusion-matrix order.
synthetic_eval = pd.DataFrame({"manual_label":["smartphone","needs_review","accessory"],"predicted_class":["needs_review","smartphone","accessory"]})
synthetic_result = evaluate_manual_labels(synthetic_eval)
assert synthetic_result["eligible_rows"] == 2
assert synthetic_result["blank_manual_labels"] == 0
assert synthetic_result["uncertain_manual_labels"] == 1
assert synthetic_result["accuracy"] == 0.5
assert synthetic_result["confusion_matrix"].index.tolist() == CLASS_ORDER
assert synthetic_result["confusion_matrix"].columns.tolist() == CLASS_ORDER
assert synthetic_result["report"].loc["feature_phone", "support"] == 0
print("Synthetic evaluation behavior checks passed")

Stratified review sample rows: 153
Predicted strata:


predicted_class
smartphone       41
feature_phone     3
accessory        26
needs_review     83
Name: count, dtype: int64

Manual labels filled: 0
Saved with human labels/notes preserved: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed/smartphone_filter_stratified_validation.csv
Stratified sample rows: 153
Definitive labels eligible: 0
Blank labels excluded: 153
Manual needs_review labels excluded: 0
No completed manual labels; evaluation cannot yet be performed.
Insufficient support classes: ['smartphone', 'feature_phone', 'accessory', 'needs_review']
Synthetic evaluation behavior checks passed


## I — Count actual review rows and J — Compare with metadata

The review file is streamed once, without loading it into pandas. This output includes only the high-rating (`rating_number >= 100`) candidates classified as smartphones. `parent_asin` is the join key; ASIN fallback is attempted only for rows missing parent ASIN, and a match is counted only if that ASIN equals a selected product identifier. Review duplicates are diagnosed but not removed.

In [114]:
smartphone_products = classification.loc[classification.predicted_class.eq("smartphone")].copy()
if smartphone_products.empty:
    raise ValueError("No smartphone candidates were classified; review count scan will not run")
smartphone_products["rating_number_numeric"] = pd.to_numeric(smartphone_products["rating_number"], errors="coerce")
selected_metadata = smartphone_products.drop_duplicates("parent_asin").set_index("parent_asin", drop=False)
selected_ids = set(selected_metadata.index.astype(str)) - {""}
if not selected_ids:
    raise ValueError("Selected smartphone products have no parent ASINs")

review_counts = Counter()
children_by_parent = defaultdict(set)
duplicate_review_keys = Counter()
exact_duplicate_rows = Counter()
seen_review_keys = set()
seen_exact_hashes = set()
review_lines = review_valid = malformed_review_rows = 0
matched_review_rows = missing_parent_rows = fallback_attempt_rows = fallback_matched_rows = 0

with gzip.open(REVIEW_PATH, "rt", encoding="utf-8") as stream:
    for line in stream:
        review_lines += 1
        try:
            review = json.loads(line)
        except json.JSONDecodeError:
            malformed_review_rows += 1
            continue
        if not isinstance(review, dict):
            malformed_review_rows += 1
            continue
        review_valid += 1
        parent = str(review.get("parent_asin") or "").strip()
        join_id = parent
        used_fallback = False
        if not parent:
            missing_parent_rows += 1
            fallback_attempt_rows += 1
            asin = str(review.get("asin") or "").strip()
            if asin in selected_ids:
                join_id = asin
                used_fallback = True
        if join_id not in selected_ids:
            continue

        matched_review_rows += 1
        if used_fallback:
            fallback_matched_rows += 1
        review_counts[join_id] += 1
        child = str(review.get("asin") or "").strip()
        if child:
            children_by_parent[join_id].add(child)
        timestamp = review.get("timestamp", review.get("sort_timestamp"))
        review_key = (join_id, child, str(review.get("user_id") or ""), str(timestamp or ""))
        if all(review_key):
            if review_key in seen_review_keys:
                duplicate_review_keys[join_id] += 1
            else:
                seen_review_keys.add(review_key)
        fingerprint = hashlib.sha256(json.dumps(review, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).digest()
        if fingerprint in seen_exact_hashes:
            exact_duplicate_rows[join_id] += 1
        else:
            seen_exact_hashes.add(fingerprint)
        if review_lines % 5_000_000 == 0:
            print(f"Review records scanned: {review_lines:,}; matched rows: {matched_review_rows:,}")

count_rows = []
for parent, product in selected_metadata.iterrows():
    actual = int(review_counts.get(parent, 0))
    metadata_count = numeric_rating_count(product.get("rating_number"))
    count_rows.append({
        "parent_asin": str(parent),
        "asin": str(product.get("asin") or ""),
        "title": product.get("title", ""),
        "rating_number": metadata_count,
        "actual_review_rows": actual,
        "count_difference": actual - metadata_count if metadata_count is not None else pd.NA,
        "actual_to_metadata_ratio": actual / metadata_count if metadata_count not in (None, 0) else pd.NA,
        "unique_child_asins": len(children_by_parent.get(parent, set())),
        "duplicate_review_key_rows": int(duplicate_review_keys.get(parent, 0)),
        "exact_duplicate_review_rows": int(exact_duplicate_rows.get(parent, 0)),
        "classifier_version": CLASSIFIER_VERSION,
    })
review_counts_df = pd.DataFrame(count_rows)
if review_counts_df.empty:
    raise ValueError("Review-count result is empty")
review_counts_df.to_csv(COUNTS_PATH, index=False)

metadata_sum = review_counts_df["rating_number"].sum(min_count=1)
actual_sum = int(review_counts_df["actual_review_rows"].sum())
valid_ratios = review_counts_df["actual_to_metadata_ratio"].dropna()
matched_products = int(review_counts_df["actual_review_rows"].gt(0).sum())
metadata_available = int(review_counts_df["rating_number"].notna().sum())
less_count = int((review_counts_df.loc[review_counts_df.rating_number.notna(), "actual_review_rows"] < review_counts_df.loc[review_counts_df.rating_number.notna(), "rating_number"]).sum())
more_count = int((review_counts_df.loc[review_counts_df.rating_number.notna(), "actual_review_rows"] > review_counts_df.loc[review_counts_df.rating_number.notna(), "rating_number"]).sum())
matched_parent_count = int(review_counts_df["actual_review_rows"].gt(0).sum())

diagnostic = pd.DataFrame([{
    "classifier_version": CLASSIFIER_VERSION,
    "selected_smartphone_products": len(review_counts_df),
    "review_lines_scanned": review_lines,
    "valid_review_records": review_valid,
    "malformed_review_records": malformed_review_rows,
    "matched_review_rows": matched_review_rows,
    "review_rows_without_parent_asin": missing_parent_rows,
    "asin_fallback_rows_attempted": fallback_attempt_rows,
    "asin_fallback_rows_matched": fallback_matched_rows,
    "products_with_at_least_one_review": matched_products,
    "products_with_zero_reviews": int((review_counts_df["actual_review_rows"] == 0).sum()),
    "review_rows_unmatched_to_selected_products": review_valid - matched_review_rows,
    "metadata_rating_number_sum": metadata_sum,
    "actual_review_row_sum": actual_sum,
    "products_with_metadata_rating_count": metadata_available,
    "actual_count_lower_than_metadata": less_count,
    "actual_count_higher_than_metadata": more_count,
    "aggregate_actual_to_metadata_ratio": actual_sum / metadata_sum if pd.notna(metadata_sum) and metadata_sum else pd.NA,
    "median_product_actual_to_metadata_ratio": valid_ratios.median() if len(valid_ratios) else pd.NA,
    "unique_child_asins": int(review_counts_df["unique_child_asins"].sum()),
    "duplicate_review_key_rows": int(review_counts_df["duplicate_review_key_rows"].sum()),
    "exact_duplicate_review_rows": int(review_counts_df["exact_duplicate_review_rows"].sum()),
    "population_definition":"Unique candidates with rating_number >= 100, phone-title retrieval match, and predicted_class=smartphone",
    "comparison_note":"rating_number is metadata product rating count; actual_review_rows counts records in this downloaded category snapshot. The values are not presumed equivalent.",
}])
diagnostic.to_csv(DIAGNOSTIC_PATH, index=False)
print("Review lines scanned:", f"{review_lines:,}")
print("Matched review rows:", f"{matched_review_rows:,}")
print("Products matched / zero:", matched_products, "/", len(review_counts_df)-matched_products)
print("Rows missing parent_asin / fallback attempted / fallback matched:", missing_parent_rows, "/", fallback_attempt_rows, "/", fallback_matched_rows)
print("Metadata rating_number sum:", metadata_sum)
print("Actual review-row sum:", actual_sum)
print("Aggregate actual-to-metadata ratio:", diagnostic.loc[0, "aggregate_actual_to_metadata_ratio"])
print("Median product-level ratio:", diagnostic.loc[0, "median_product_actual_to_metadata_ratio"])
print("Duplicate review-key rows (reported, not removed):", int(review_counts_df["duplicate_review_key_rows"].sum()))
print("Exact duplicate review rows (reported, not removed):", int(review_counts_df["exact_duplicate_review_rows"].sum()))
print("Saved:", COUNTS_PATH)
print("Saved:", DIAGNOSTIC_PATH)
display(diagnostic.T)
display(review_counts_df.sort_values("actual_review_rows", ascending=False).head(20))

Review lines scanned: 20,812,945
Matched review rows: 325,986
Products matched / zero: 2768 / 0
Rows missing parent_asin / fallback attempted / fallback matched: 0 / 0 / 0
Metadata rating_number sum: 1727338.0
Actual review-row sum: 325986
Aggregate actual-to-metadata ratio: 0.18872160515197373
Median product-level ratio: 0.1702127659574468
Duplicate review-key rows (reported, not removed): 4090
Exact duplicate review rows (reported, not removed): 4090
Saved: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed/smartphone_review_counts.csv
Saved: /Users/vaishnavipoojary/CliO-customer-feedback-analytics/data/processed/smartphone_review_count_diagnostic.csv


,0
classifier_version,notebook03-rule-v1.2
selected_smartphone_products,2768
review_lines_scanned,20812945
valid_review_records,20812945
malformed_review_records,0
matched_review_rows,325986
review_rows_without_parent_asin,0
asin_fallback_rows_attempted,0
asin_fallback_rows_matched,0
products_with_at_least_one_review,2768


,parent_asin,asin,title,rating_number,actual_review_rows,count_difference,actual_to_metadata_ratio,unique_child_asins,duplicate_review_key_rows,exact_duplicate_review_rows,classifier_version
58,B086RM3DQS,,"Apple iPhone 8, US Version, 256GB, Space Gray ...",51290.0,7266,-44024.0,0.141665,50,77,77,notebook03-rule-v1.2
1684,B01BZC262Q,,iOttie Easy One Touch 3 IPhone XS MAX R 8 Plus...,33385.0,5016,-28369.0,0.150247,1,23,23,notebook03-rule-v1.2
24,B08D969C67,,"Apple iPhone X, US Version, 256GB, Silver - AT...",23128.0,4607,-18521.0,0.199196,29,60,60,notebook03-rule-v1.2
133,B01MPXRXR0,,BLU Advance 5.5 HD -Unlocked Dual Sim Smartpho...,7533.0,4013,-3520.0,0.532723,5,52,52,notebook03-rule-v1.2
466,B00TW47DSW,,"Motorola Nexus 6 GSM Unlocked Smartphone, 32GB...",3845.0,3285,-560.0,0.854356,6,30,30,notebook03-rule-v1.2
2434,B008GVL9YQ,,iXCC 10ft Extra Long 30 Pin to USB SYNC and Ch...,3945.0,3148,-797.0,0.797972,1,24,24,notebook03-rule-v1.2
284,B019BX7F86,,"Huawei Nexus 6P unlocked smartphone, 32GB Gold...",2746.0,2328,-418.0,0.847779,11,76,76,notebook03-rule-v1.2
1695,B00A29WCA0,,Samsung Galaxy S3 Mini GT-i8190 GSM Unlocked I...,3561.0,2214,-1347.0,0.621735,1,23,23,notebook03-rule-v1.2
663,B07RWPFQLM,,Google - Pixel 3a with 64GB Memory Cell Phone ...,8637.0,2134,-6503.0,0.247077,7,19,19,notebook03-rule-v1.2
104,B07GYY6VTD,,Moto G6 with Alexa Hands-Free – 32 GB – Unlock...,717.0,2130,1413.0,2.970711,3,32,32,notebook03-rule-v1.2


## K — Reproducibility and checks

The product-level join uses parent ASIN consistently with metadata parent identifiers. The official [Amazon Reviews 2023 documentation](https://amazon-reviews-2023.github.io/main.html) defines `rating_number` as the number of ratings in the product, and recommends `parent_asin` for matching metadata across variants. Its category summary reports 20.8M Cell Phones and Accessories reviews; the local 20,812,945-row scan is consistent with that rounded figure. The review JSONL contains individual review records; the two counts may cover different snapshots or aggregation scopes. This comparison alone cannot establish the precise cause of any difference. Duplicate review rows are counted as evidence and retained in totals.

In [115]:
# Data-integrity assertions that do not require another full input scan.
assert set(classification["predicted_class"].unique()).issubset(CLASS_ORDER)
assert classification["parent_asin"].map(lambda x: isinstance(x, str)).all()
assert review_counts_df["parent_asin"].is_unique
assert review_counts_df["actual_to_metadata_ratio"].notna().equals(review_counts_df["rating_number"].notna() & review_counts_df["rating_number"].ne(0))
assert independent_sample is None or independent_sample["manual_label"].equals(pd.read_csv(INDEPENDENT_PATH)["manual_label"])
assert stratified_sample["manual_label"].equals(pd.read_csv(STRATIFIED_PATH)["manual_label"])
assert stratified_sample["reviewer_notes"].fillna("").equals(pd.read_csv(STRATIFIED_PATH)["reviewer_notes"].fillna(""))
print("Integrity checks passed: valid class outputs, unique review-count IDs, safe ratios, and preserved manual labels/notes.")

Integrity checks passed: valid class outputs, unique review-count IDs, safe ratios, and preserved manual labels/notes.


## Manual work remaining and interpretation

The existing independent sample currently contains human labels and is scored above. The stratified review sample keeps `manual_label` and `reviewer_notes` blank unless you fill them. Enter only one of `smartphone`, `feature_phone`, `accessory`, or `needs_review` in `manual_label`; keep notes in `reviewer_notes`, save the CSV, then rerun Sections G–H to score it. Do not use the predictions as ground truth.

No high overall accuracy alone establishes production readiness. Inspect class support and mismatches, especially because the existing validation labels are small and class-imbalanced. The review-count comparison is descriptive: do not interpret its ratio as review completeness without independent evidence.